| **Field**              | **Information** |
|------------------------|-----------------|
| **Name**               |      Ayesha Ameer           |
| **Registration Number**|      04102213021           |
| **Project Title**      |       GenAI Domain Assistant - RAG (Retrieval Augmented Generation)         |
| **Week**               |        Week 9         |
| **Instructor**         |      Sir Faiz Ahmad           |

In [1]:
pip install langchain langchain-openai pypdf  langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install -U langchain langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# PART 1: DOCUMENT LOADING

## Task 1.1: Load Documents

In [9]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Load environment variables
load_dotenv()

# =========================
# Load documents from folder
# =========================
loader = DirectoryLoader(
    path="company_docs/",
    glob="*.txt",
    loader_cls=lambda path: TextLoader(path, encoding="utf-8")
)

documents = loader.load()

print(f"\n✅ Loaded {len(documents)} documents\n")

if documents:
    print("📄 First document preview:\n")
    print(documents[0].page_content[:300])
else:
    print("⚠️ No documents found in the directory.")

'''# =========================
# (Optional) Split documents
# =========================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"\n✂️ Total chunks created: {len(chunks)}")'''


✅ Loaded 3 documents

📄 First document preview:

Company Benefits Policy

Health Insurance:
Fully covered medical insurance for all full-time employees.

Retirement Plan:
401(k) matching up to 5% of employee contributions.

Wellness Benefits:
Annual wellness stipend for gym or fitness activities.

Learning & Development:
Employees receive yearly b


'# =========================\n# (Optional) Split documents\n# =========================\ntext_splitter = RecursiveCharacterTextSplitter(\n    chunk_size=500,\n    chunk_overlap=50\n)\n\nchunks = text_splitter.split_documents(documents)\n\nprint(f"\n✂️ Total chunks created: {len(chunks)}")'

## Task 1.2: Split into Chunks

In [10]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # characters per chunk
    chunk_overlap=50,     # overlap for context continuity
    length_function=len,
    separators=[
        "\n\n",   # paragraph breaks (highest priority)
        "\n",     # line breaks
        ". ",     # sentence endings
        " ",      # words
        ""        # fallback character-level split
    ]
)

# Split documents safely
chunks = text_splitter.split_documents(documents)

# Output results
print(f"Split into {len(chunks)} chunks\n")

print("Sample chunks:\n")

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:")
    print("-" * 40)
    print(chunk.page_content.strip())
    print(f"\nLength: {len(chunk.page_content)} chars")
    print("=" * 60)

Split into 3 chunks

Sample chunks:

Chunk 1:
----------------------------------------
Company Benefits Policy

Health Insurance:
Fully covered medical insurance for all full-time employees.

Retirement Plan:
401(k) matching up to 5% of employee contributions.

Wellness Benefits:
Annual wellness stipend for gym or fitness activities.

Learning & Development:
Employees receive yearly budget for courses and certifications.

Length: 337 chars
Chunk 2:
----------------------------------------
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Leave:
12 weeks paid parental leave for primary caregivers.
6 weeks paid leave for secondary caregivers.

Length: 402 chars
Chunk 3:
----------------------------------------
IT & Security Policy

Password Pol

# PART 2: SIMPLE RETRIEVAL

## Task 2.1: Build Keyword Search

In [12]:
def simple_search(query, chunks, top_k=3):
    """
    Simple keyword-based search using frequency scoring
    """

    query_tokens = query.lower().split()
    scored_chunks = []

    for chunk in chunks:
        content = chunk.page_content.lower()

        # score based on keyword frequency
        score = sum(content.count(token) for token in query_tokens)

        if score > 0:
            scored_chunks.append((score, chunk))

    # sort by score descending
    scored_chunks.sort(key=lambda x: x[0], reverse=True)

    return [chunk for score, chunk in scored_chunks[:top_k]]


# =========================
# TEST SEARCH
# =========================
query = "What is the vacation policy?"
relevant_chunks = simple_search(query, chunks)

print(f"\nFound {len(relevant_chunks)} relevant chunks:\n")

for i, chunk in enumerate(relevant_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content.strip())
    print("-" * 50)


Found 2 relevant chunks:

--- Chunk 1 ---
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Leave:
12 weeks paid parental leave for primary caregivers.
6 weeks paid leave for secondary caregivers.
--------------------------------------------------
--- Chunk 2 ---
IT & Security Policy

Password Policy:
Employees must use strong passwords and change them every 90 days.

Device Usage:
Company devices should be used for work-related tasks only.

Data Security:
Do not share confidential company data externally without approval.

Software Installation:
Only approved software may be installed on company systems.
--------------------------------------------------


## Task 2.2: Test Different Queries

In [13]:
# =========================
# TEST MULTIPLE QUERIES
# =========================

test_queries = [
    "How many vacation days do employees get?",
    "What is the remote work policy?",
    "Tell me about parental leave",
]

for query in test_queries:
    print("\n" + "=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    results = simple_search(query, chunks, top_k=2)

    print(f"Relevant chunks found: {len(results)}\n")

    if results:
        print("TOP RESULT PREVIEW:\n")
        print(results[0].page_content.strip()[:200] + "...\n")

        # Optional: show all results
        for i, chunk in enumerate(results):
            print(f"--- Result {i+1} ---")
            print(chunk.page_content.strip()[:300])
            print("-" * 50)
    else:
        print("No relevant chunks found.")


QUERY: How many vacation days do employees get?
Relevant chunks found: 2

TOP RESULT PREVIEW:

Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Em...

--- Result 1 ---
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Le
--------------------------------------------------
--- Result 2 ---
IT & Security Policy

Password Policy:
Employees must use strong passwords and change them every 90 days.

Device Usage:
Company devices should be used for work-related tasks only.

Data Security:
Do not share confidential company data externally without approval.

Software Installation:
Only approv
------------

# PART 3: RAG PIPELINE

## Task 3.1: Build RAG Function

In [28]:
from openai import OpenAI
import os

# =========================
# OPENROUTER SETUP
# =========================

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# =========================
# LLM CALL FUNCTION
# =========================
def call_llm(prompt):
    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=500
    )
    return response.choices[0].message.content


# =========================
# RAG FUNCTION
# =========================
def rag_query(query, chunks, top_k=3):

    # Step 1: Retrieve
    relevant_chunks = simple_search(query, chunks, top_k)

    if not relevant_chunks:
        return "No relevant information found in documents."

    # Step 2: Context building
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{chunk.page_content.strip()}"
        for i, chunk in enumerate(relevant_chunks)
    )

    # Step 3: Prompt
    prompt = f"""
You are a helpful assistant.

RULES:
- Use ONLY the provided context.
- If answer is not in context, say: "Not available in the documents."
- Do not hallucinate.

CONTEXT:
----------------
{context}
----------------

QUESTION:
{query}

ANSWER:
""".strip()

    # Step 4: Generate
    return call_llm(prompt)

## Task 3.2: Test RAG System

In [29]:
# =========================
# TEST RAG SYSTEM
# =========================

questions = [
    "How many vacation days do full-time employees get?",
    "Can employees work from home?",
    "What is the parental leave policy?",
    "What is the dress code?"  # intentionally out-of-doc question
]

for i, question in enumerate(questions, 1):
    print("\n" + "=" * 80)
    print(f"QUESTION {i}: {question}")
    print("=" * 80)

    try:
        answer = rag_query(question, chunks)

        print("\nANSWER:")
        print("-" * 80)
        print(answer)
        print("-" * 80)

    except Exception as e:
        print("\nERROR occurred while processing this question:")
        print(str(e))


QUESTION 1: How many vacation days do full-time employees get?

ANSWER:
--------------------------------------------------------------------------------
Full-time employees receive 15 days of paid vacation per year.
--------------------------------------------------------------------------------

QUESTION 2: Can employees work from home?

ANSWER:
--------------------------------------------------------------------------------
Yes, employees may work remotely up to 3 days per week, but remote work requires manager approval.
--------------------------------------------------------------------------------

QUESTION 3: What is the parental leave policy?

ANSWER:
--------------------------------------------------------------------------------
The parental leave policy provides 12 weeks of paid parental leave for primary caregivers and 6 weeks of paid leave for secondary caregivers.
--------------------------------------------------------------------------------

QUESTION 4: What is the dre

# BONUS: COMPARE WITH VS WITHOUT RAG 

In [30]:
# =========================
# ASK WITHOUT RAG (DIRECT LLM)
# =========================
def ask_without_rag(question):
    """
    Ask LLM directly (no document context)
    """

    messages = [
        {
            "role": "system",
            "content": "You are a helpful HR assistant."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages,
        temperature=0.7,
        max_tokens=500
    )

    return response.choices[0].message.content


# =========================
# COMPARE RAG VS NO RAG
# =========================
question = "How many vacation days do employees get?"

print("\n" + "=" * 80)
print("WITHOUT RAG")
print("=" * 80)
print(ask_without_rag(question))


print("\n" + "=" * 80)
print("WITH RAG")
print("=" * 80)
print(rag_query(question, chunks))


WITHOUT RAG
The number of vacation days employees receive can vary widely depending on several factors, including the company's policies, the employee's length of service, and local labor laws. 

In the United States, for example, many companies offer between 10 to 20 vacation days per year for full-time employees. Some organizations may provide additional days for long-term employees or offer unlimited vacation policies.

In other countries, vacation entitlements can be mandated by law. For instance, many European countries require a minimum of four weeks of paid vacation per year.

To get the specific vacation policy for a particular company, it's best to refer to the employee handbook or consult with the HR department.

WITH RAG
Employees receive 15 days of paid vacation per year.
